In [90]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import re

from tqdm import trange, tqdm

x = np.load('x.npy')
x = np.clip(x, 0, 26)

y = np.load('y.npy')
y = np.clip(y, 0, 26)

In [91]:
GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [92]:
LEN = len(x)
print(LEN)
split = int(0.8*LEN)

x_train = torch.tensor(x[:split])
y_train = torch.tensor(y[:split])
x_test = torch.tensor(x[split:])
y_test = torch.tensor(y[split:])

51010


In [93]:
class ShakeText(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [94]:
batch_size = 64
train_loader = DataLoader(ShakeText(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(ShakeText(x_test, y_test), batch_size=batch_size, shuffle=False)

In [95]:
X, Y = next(iter(train_loader))

print(X.shape)
print(Y.shape)

torch.Size([64, 100])
torch.Size([64, 5])


In [96]:
bidirec = True

class rnn(nn.Module) :
    def __init__ (self, vocab_size, input_size, hidden_size, output_size, pad_idx):
        super(rnn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size, pad_idx)
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True, num_layers=1, bidirectional=bidirec)
        self.fc = nn.Linear((1+int(bidirec))*hidden_size, output_size*vocab_size)

    def forward (self, x):
        x = self.embedding(x)
        _, (h_n,_) = self.rnn(x) # replace underscore with a pair of underscores for LSTM
        if bidirec:
            h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            h = h_n[-1]
        out = self.fc(h)
        return out.view(-1, 5, 27)

In [97]:
model = rnn(27, 64, 1024, 5, 0).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
n_epochs = 50
print(model)

rnn(
  (embedding): Embedding(27, 64, padding_idx=0)
  (rnn): LSTM(64, 1024, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=2048, out_features=135, bias=True)
)


In [98]:
def train_epoch(model, train_loader, criterion, optimizer, loss_logger):
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        
        optimizer.zero_grad()
        outputs = model(data.to(device))
        loss = criterion(outputs.permute(0,2,1), target.to(device).long())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        loss_logger.append(loss.item())

    return model, optimizer, loss_logger

def test_model(model, test_loader, criterion, loss_logger):
    with torch.no_grad():
        correct_predictions = 0
        total_predictions = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(test_loader, desc="Testing", leave=False)):   
            
            outputs = model(data.to(device))        
            _, predicted = torch.max(outputs, dim=-1)
            correct_predictions += (predicted == target.to(device)).sum().item()
            total_predictions += target.numel()
            loss = criterion(outputs.permute(0,2,1), target.to(device).long())
            loss_logger.append(loss.item())
            
        acc = (correct_predictions/total_predictions) * 100.0
        
        return loss_logger, acc

train_loss = []
test_loss  = []
test_acc   = []

In [99]:
for i in trange(n_epochs, desc="Epoch", leave=False):
    model, optimizer, train_loss = train_epoch(model, train_loader, criterion, optimizer, train_loss)
    
    test_loss, acc = test_model(model, test_loader, criterion, test_loss)
    test_acc.append(acc)

print("Final Accuracy: %.2f%%" % acc)

Final Accuracy: 33.26%
